In [ ]:
!pip install -U datasets pyldavis spacy nltk
!python -m spacy download es_core_news_sm

In [ ]:
import re
import unicodedata
from collections import Counter
from textwrap import shorten

import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import pyLDAvis
from IPython.display import display

# Stemming (Spanish)
from nltk.stem.snowball import SnowballStemmer
import warnings
warnings.filterwarnings("ignore")


# OpenAssistant Conversations Dataset (OASST1)

## A Deep Dive for the LDA Topic Modeling Notebook

---

## 1. What is OASST1?

**OpenAssistant Conversations (OASST1)** is a large-scale, human-generated, human-annotated conversation dataset released by [LAION](https://laion.ai/) in April 2023. It was created to **democratize research on large language model alignment** — the process of making AI assistants helpful, harmless, and honest.

### Key Statistics

| Metric | Value |
|--------|-------|
| Total messages | **161,443** |
| Conversation trees | **66,497** (all) / **10,364** (curated) |
| Languages | **35** |
| Quality ratings | **461,292** |
| Volunteers worldwide | **13,500+** |
| HuggingFace train split | 84,437 messages |
| HuggingFace validation split | 4,401 messages |

---

## 2. Why Was It Created?

Before OASST1, high-quality conversational data for training AI assistants was:

1. **Proprietary** — locked inside companies like OpenAI, Google, Anthropic
2. **Expensive** — professional annotation costs millions of dollars
3. **Limited in languages** — predominantly English

The OpenAssistant project aimed to create a **free, open-source alternative** that anyone could use to train their own ChatGPT-like assistant.

### The Vision

> "We want to build the assistant of the future, able to not only write email and cover letters, but do meaningful work, use APIs, dynamically research information, and much more, with the ability to be personalized and extended by anyone."
> — OpenAssistant GitHub

---

## 3. How Was the Data Collected?

### Crowdsourcing Methodology

The data was collected via a **web application** at [open-assistant.io](https://open-assistant.io) where volunteers could:

```
┌─────────────────────────────────────────────────────────────────┐
│                    DATA COLLECTION TASKS                        │
├─────────────────────────────────────────────────────────────────┤
│  1. CREATE INITIAL PROMPTS                                      │
│     → Write a question/request as if talking to an AI assistant │
│                                                                 │
│  2. WRITE ASSISTANT RESPONSES                                   │
│     → Reply to prompts as if you were a helpful AI              │
│                                                                 │
│  3. RANK RESPONSES                                              │
│     → Compare multiple assistant replies and rank best → worst  │
│                                                                 │
│  4. LABEL MESSAGES                                              │
│     → Rate quality, toxicity, helpfulness, creativity, etc.     │
└─────────────────────────────────────────────────────────────────┘
```

### Tree State Machine

Each conversation follows a **state machine** that ensures quality:

```
                    ┌──────────────────┐
                    │  Initial Prompt  │
                    │     Review       │
                    └────────┬─────────┘
                             │
              ┌──────────────┴──────────────┐
              │                             │
              ▼                             ▼
    ┌─────────────────┐          ┌──────────────────┐
    │  Aborted (Low   │          │     Growing      │
    │    Grade)       │          │   (Collecting    │
    └─────────────────┘          │    responses)    │
                                 └────────┬─────────┘
                                          │
                                          ▼
                                 ┌──────────────────┐
                                 │  Ready for       │
                                 │  Export          │
                                 └──────────────────┘
```

### Quality Control

Messages were filtered based on:
- **Spam detection** — removing promotional or nonsensical content
- **Language mismatch** — ensuring the declared language matches content
- **PII detection** — removing personal identifiable information
- **Hate speech / Sexual content** — flagging inappropriate material
- **Human review** — multiple reviewers per message

---

## 4. Data Structure

### Message Schema

Each message in the dataset contains these fields:

```python
{
    # ═══════════════════════════════════════════════════════════
    # IDENTIFIERS
    # ═══════════════════════════════════════════════════════════
    "message_id": "6ab24d72-0181-...",      # Unique ID for this message
    "parent_id": None,                       # ID of parent message (None = root prompt)
    "message_tree_id": "6ab24d72-...",       # ID of the conversation tree
    "user_id": "c3fe8c76-...",               # Anonymous contributor ID
    
    # ═══════════════════════════════════════════════════════════
    # CONTENT
    # ═══════════════════════════════════════════════════════════
    "text": "Can you write a short intro...", # The actual message content
    "role": "prompter",                       # "prompter" (user) or "assistant"
    "lang": "en",                             # ISO language code
    
    # ═══════════════════════════════════════════════════════════
    # METADATA
    # ═══════════════════════════════════════════════════════════
    "created_date": "2023-02-05T14:23:50...",
    "review_count": 3,                        # Number of human reviews
    "review_result": True,                    # Passed review?
    "deleted": False,                         # Was it removed?
    "synthetic": False,                       # AI-generated? (we filter these out)
    "model_name": None,                       # If synthetic, which model?
    "rank": 0,                                # Rank among sibling responses (0 = best)
    
    # ═══════════════════════════════════════════════════════════
    # TOXICITY SCORES (from Detoxify model)
    # ═══════════════════════════════════════════════════════════
    "detoxify": {
        "toxicity": 0.00044,
        "severe_toxicity": 0.00003,
        "obscene": 0.00023,
        "identity_attack": 0.00014,
        "insult": 0.00039,
        "threat": 0.00004,
        "sexual_explicit": 0.00002
    },
    
    # ═══════════════════════════════════════════════════════════
    # HUMAN LABELS (crowdsourced quality ratings)
    # ═══════════════════════════════════════════════════════════
    "labels": [
        {"name": "spam",           "value": 0.0,   "count": 3},
        {"name": "lang_mismatch",  "value": 0.0,   "count": 3},
        {"name": "quality",        "value": 0.917, "count": 3},
        {"name": "helpfulness",    "value": 0.75,  "count": 2},
        {"name": "creativity",     "value": 0.667, "count": 3},
        {"name": "humor",          "value": 0.333, "count": 3},
        {"name": "toxicity",       "value": 0.167, "count": 3},
        {"name": "violence",       "value": 0.0,   "count": 3},
        # ... etc
    ],
    
    # ═══════════════════════════════════════════════════════════
    # EMOJI REACTIONS
    # ═══════════════════════════════════════════════════════════
    "emojis": [
        {"name": "+1", "count": 10},
        {"name": "_skip_reply", "count": 1}
    ]
}
```

### Conversation Tree Structure

Messages form **tree structures**, not linear conversations:

```
                     [Initial Prompt]
                     "What is monopsony?"
                            │
           ┌────────────────┼────────────────┐
           │                │                │
           ▼                ▼                ▼
    [Assistant A]    [Assistant B]    [Assistant C]
    (rank: 0 ⭐)     (rank: 1)        (rank: 2)
           │                │
           ▼                ▼
    [User Follow-up]  [User Follow-up]
    "Explain more"    "Give examples"
           │
           ▼
    [Assistant Reply]
```

This tree structure allows:
- **Multiple response options** per prompt (for preference learning)
- **Branching conversations** (different follow-up paths)
- **Ranking data** for training reward models (RLHF)

---

## 5. Example: What Text Goes Into LDA

After filtering and preprocessing, here are examples of **actual Spanish root prompts** that our LDA model analyzes:

### 5.1 Raw Text (Before Preprocessing)

These are real `df["text"]` values after filtering for Spanish root prompts:

```
1. "¿Cuáles son las etapas del desarrollo y en qué consisten según Piaget?"

2. "Escribe un poema sobre la soledad en una noche de invierno"

3. "¿Cómo puedo ordenar una lista en Python de mayor a menor?"

4. "Explícame qué es la inteligencia artificial como si tuviera 5 años"

5. "Dame una receta fácil para hacer tacos al pastor en casa"

6. "¿Cuál fue la causa principal de la Segunda Guerra Mundial?"

7. "¿Qué diferencia hay entre machine learning y deep learning?"

8. "Ayúdame a escribir un correo formal para solicitar un aumento de sueldo"
```

### 5.2 Cleaned Text (After Preprocessing)

After running `preprocess()` (lowercase → remove URLs → strip accents → tokenize → remove stopwords → stem):

```
1. "etap desarroll consist piaget"

2. "poem soled noch inviern"

3. "orden list python mayor menor"

4. "inteligent artificial años"

5. "recet facil tac pastor cas"

6. "caus principal segund guerr mundial"

7. "diferent machine learn deep learn"

8. "escrib corre formal solicit aument sueld"
```

### 5.3 What LDA Sees

LDA receives a **bag-of-words matrix** where each row is a document and each column is a term count:

```
              etap  desarroll  piaget  poem  soled  python  orden  recet  guerr  ...
prompt_1      [1      1         1       0     0      0       0      0      0    ...]
prompt_2      [0      0         0       1     1      0       0      0      0    ...]
prompt_3      [0      0         0       0     0      1       1      0      0    ...]
prompt_4      [0      0         0       0     0      0       0      0      0    ...]
prompt_5      [0      0         0       0     0      0       0      1      0    ...]
prompt_6      [0      0         0       0     0      0       0      0      1    ...]
   ⋮
```

From this matrix, LDA discovers latent topics by finding groups of words that frequently co-occur across documents.

---

## 6. Language Distribution

The dataset is **multilingual** but heavily skewed toward English and Spanish:

```
Language Distribution (Top 10)
══════════════════════════════════════════════════════════
English (en)     ████████████████████████████████████  52%
Spanish (es)     ██████████████████                    26%
Russian (ru)     ████                                   6%
German (de)      ███                                    4%
French (fr)      ██                                     3%
Chinese (zh)     ██                                     3%
Portuguese (pt)  █                                      2%
Italian (it)     █                                      1%
Polish (pl)      █                                      1%
Other (25 langs) █                                      2%
══════════════════════════════════════════════════════════
```

### Why So Much Spanish?

The high proportion of Spanish content is attributed to **prominent figures in the Spanish-speaking ML community** promoting the project, leading to significant volunteer participation from Latin America and Spain.

**This makes OASST1 particularly valuable for Spanish NLP research** — which is why we filter for Spanish prompts in our LDA notebook.

---

## 7. Why We Apply These Filters in the Notebook

```python
df = df[df["role"].astype(str).str.lower().isin(USER_ROLES)]
df = df[df["parent_id"].isna()]
df = df[df["deleted"] == False]
df = df[df["synthetic"] == False]
df = df[df["lang"].astype(str).str.lower().str.startswith("es")]
```

| Filter | Reason | Impact |
|--------|--------|--------|
| `role in USER_ROLES` | We analyze **what users ask**, not AI responses | ~50% reduction |
| `parent_id.isna()` | Only **initial prompts** (root messages), not follow-ups | ~70% reduction |
| `deleted == False` | Remove low-quality/inappropriate content | ~5% reduction |
| `synthetic == False` | Only **human-written** prompts, no AI-generated | <1% reduction |
| `lang == "es"` | Focus on **Spanish** for language-specific analysis | ~74% reduction |

After filtering, we typically get **~1,500-3,000 Spanish root prompts** — enough for meaningful topic modeling.

---

## 8. Types of Prompts in the Dataset

Based on the Spanish subset, common prompt categories include:

### Knowledge & Explanation
```
"¿Cuáles son las etapas del desarrollo según Piaget?"
"Explícame qué es la fotosíntesis"
"¿Qué diferencia hay entre hardware y software?"
```

### Creative Writing
```
"Escribe un poema sobre la soledad"
"Inventa una historia corta de ciencia ficción"
"Dame ideas para un nombre de empresa"
```

### Technical Assistance
```
"¿Cómo puedo ordenar una lista en Python?"
"Explica cómo funciona una red neuronal"
"¿Qué es Docker y para qué sirve?"
```

### Advice & Recommendations
```
"¿Qué libro me recomiendas para aprender economía?"
"Dame consejos para una entrevista de trabajo"
"¿Cuál es la mejor forma de ahorrar dinero?"
```

### Translation & Language
```
"Traduce esta frase al inglés"
"¿Cómo se dice 'buenos días' en japonés?"
"Corrige los errores gramaticales de este texto"
```

---

## 9. Relevance for Topic Modeling

OASST1 is **ideal for LDA topic modeling** because:

1. **Diverse topics** — Users asked about everything from science to cooking to philosophy
2. **Natural language** — Real human prompts, not synthetic templates
3. **Clean metadata** — Language tags, quality scores, and spam filtering
4. **Sufficient volume** — Thousands of prompts per major language
5. **Open license** — Apache 2.0, free for research and commercial use

### What LDA Will Discover

By running LDA on Spanish prompts, we expect to find topics like:

| Topic ID | Likely Theme | Example Terms |
|----------|--------------|---------------|
| 0 | Programming | python, código, función, programa, variable |
| 1 | History | guerra, historia, país, mundial, rey |
| 2 | Cooking | receta, cocina, ingrediente, comida, preparar |
| 3 | Science | energía, célula, átomo, física, química |
| 4 | Writing | texto, escribir, historia, personaje, cuento |
| 5 | Health | salud, enfermedad, médico, síntoma, tratamiento |
| ... | ... | ... |

---

## 10. Dataset Files Available

On HuggingFace, multiple formats are available:

| File | Contents | Use Case |
|------|----------|----------|
| `oasst_ready.trees.jsonl.gz` | 10,364 curated trees | SFT & Reward Model training |
| `oasst_ready.messages.jsonl.gz` | 88,838 messages (flat) | General analysis |
| `oasst_all.trees.jsonl.gz` | 66,497 all trees | Research (includes low-quality) |
| `oasst_all.messages.jsonl.gz` | 161,443 messages | Complete dataset |
| `train/validation parquet` | HF Datasets format | Easy loading with `load_dataset()` |

---

## 11. Citation

If using this dataset in academic work:

```bibtex
@article{kopf2023openassistant,
  title={OpenAssistant Conversations -- Democratizing Large Language Model Alignment},
  author={K{\"o}pf, Andreas and Kilcher, Yannic and von R{\"u}tte, Dimitri and others},
  journal={arXiv preprint arXiv:2304.07327},
  year={2023}
}
```

---

## 12. Summary

| Aspect | Description |
|--------|-------------|
| **What** | Human-generated conversation dataset for AI assistant training |
| **Who** | LAION + 13,500 volunteers worldwide |
| **When** | Released April 2023 |
| **Size** | 161K messages, 35 languages, 460K+ quality ratings |
| **Purpose** | Democratize LLM alignment research |
| **License** | Apache 2.0 (free for any use) |
| **Spanish data** | ~26% of corpus — excellent for Spanish NLP |
| **Our use** | Extract Spanish root prompts → LDA topic modeling |

---

## Further Resources

- 📄 [Paper on arXiv](https://arxiv.org/abs/2304.07327)
- 🤗 [Dataset on HuggingFace](https://huggingface.co/datasets/OpenAssistant/oasst1)
- 💻 [GitHub Repository](https://github.com/LAION-AI/Open-Assistant)
- 📓 [Getting Started Notebook](https://github.com/LAION-AI/Open-Assistant/blob/main/notebooks/openassistant-oasst1/getting-started.ipynb)

In [ ]:


# -----------------------
# 1) Load dataset
# -----------------------
ds = load_dataset("OpenAssistant/oasst1")
df = pd.concat([ds["train"].to_pandas(), ds["validation"].to_pandas()], ignore_index=True)



In [ ]:


# -----------------------
# 2) Filter: Spanish root prompts from the user
# -----------------------
USER_ROLES = {"user", "prompter", "human"}

df = df[df["role"].astype(str).str.lower().isin(USER_ROLES)]
df = df[df["parent_id"].isna()]                 # root message in the thread
df = df[df["deleted"] == False]
df = df[df["synthetic"] == False]
df = df[df["lang"].astype(str).str.lower().str.startswith("es")]

# Optional: sample for faster teaching demos
RANDOM_SEED = 42

docs_raw = df["text"].astype(str).tolist()
print("Docs (raw):", len(docs_raw))


In [ ]:


# -----------------------
# 3) Preprocessing (standard NLP pipeline + stemming)
#    clean -> normalize -> tokenize -> stopwords -> stem -> re-join
# -----------------------
def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

STOPWORDS_ES = {
    # common Spanish
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","no",
    "una","su","al","lo","como","mas","pero","sus","le","ya","o","este","si","porque",
    "esta","entre","cuando","muy","sin","sobre","tambien","me","hasta","hay","donde",
    "quien","desde","todo","nos","durante","todos","uno","les","ni","contra","otros",

    # question/prompt words (important for chatbot prompts)
    "que","como","cual","cuales","quien","donde","cuando","cuanto","cuantos","por","porque",

    # prompt fluff
    "hola","buenas","gracias","porfavor","favor",
    "puedes","podrias","ayudame","necesito","dime","explica","explicame","describe","resume",
    "haz","dame","escribe","crea","genera",

    # common high-frequency verbs/forms
    "ser","estar","tener","hacer","poder",
    "soy","eres","es","son","estoy","esta","estan","tengo","tiene","tienen","quiero","puedo",
}

# Normalize stopwords in the same way we normalize text
STOPWORDS_ES = {strip_accents(w.lower()) for w in STOPWORDS_ES}

# Spanish stemmer (simple + no extra downloads)
stemmer = SnowballStemmer("spanish")

def preprocess(text: str) -> str:
    # A) CLEANING (remove obvious noise)
    text = (text or "").lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # remove URLs

    # B) NORMALIZATION (accent folding)
    text = strip_accents(text)  # "qué" -> "que"

    # C) TOKENIZATION (letters only; after accent stripping)
    tokens = re.findall(r"[a-zñ]+", text)

    # D) FILTERING (stopwords + short tokens)
    tokens = [t for t in tokens if len(t) >= 3 and t not in STOPWORDS_ES]

    # E) STEMMING (reduce words to their root)
    tokens = [stemmer.stem(t) for t in tokens]

    # Return a string for CountVectorizer
    return " ".join(tokens)

docs_clean = [preprocess(t) for t in docs_raw]

# Keep alignment between raw and clean by filtering pairs together
pairs = [(r, c) for r, c in zip(docs_raw, docs_clean) if c.strip()]
docs_raw, docs_clean = zip(*pairs) if pairs else ([], [])
docs_raw, docs_clean = list(docs_raw), list(docs_clean)

print("Docs (clean, non-empty):", len(docs_clean))
print("Top tokens after preprocessing:", Counter(" ".join(docs_clean).split()).most_common(20))



In [ ]:

# -----------------------
# 4) Bag-of-words
# -----------------------
vectorizer = CountVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2))
X = vectorizer.fit_transform(docs_clean)
vocab = np.array(vectorizer.get_feature_names_out())
print("Vocab size:", len(vocab))



In [ ]:

# -----------------------
# 5) Fit LDA
# -----------------------
n_topics = 5
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method="batch",
    random_state=RANDOM_SEED,
    max_iter=25
)
lda.fit(X)


In [ ]:


# -----------------------
# 6) Topics: print + DataFrame of top words per topic
# -----------------------
def show_topics(model, vocab, top_n=12):
    for k, weights in enumerate(model.components_):
        top_idx = np.argsort(weights)[::-1][:top_n]
        top_terms = [vocab[i] for i in top_idx]
        print(f"Topic {k}: {', '.join(top_terms)}")

def topics_dataframe(model, vocab, top_n=12):
    rows = []
    for topic_id, weights in enumerate(model.components_):
        top_idx = np.argsort(weights)[::-1][:top_n]
        for rank, i in enumerate(top_idx, start=1):
            rows.append({
                "topic_id": topic_id,
                "rank": rank,
                "term": vocab[i],
                "weight": float(weights[i]),
            })
    return pd.DataFrame(rows)

show_topics(lda, vocab, top_n=12)

df_topics = topics_dataframe(lda, vocab, top_n=12)



In [ ]:

# -----------------------
# 7) Documents: DataFrame with topic assignment + probabilities
# -----------------------
doc_topic = lda.transform(X)
top_topic = doc_topic.argmax(axis=1)
top_conf = doc_topic.max(axis=1)

df_docs = pd.DataFrame({
    "doc_id": np.arange(len(docs_raw)),
    "text_raw": [shorten(t, width=160, placeholder="…") for t in docs_raw],
    "text_clean": [shorten(t, width=160, placeholder="…") for t in docs_clean],
    "top_topic": top_topic,
    "top_topic_conf": np.round(top_conf, 3),
})

for k in range(n_topics):
    df_docs[f"topic_{k}_prob"] = np.round(doc_topic[:, k], 3)


# -----------------------
# 8) pyLDAvis (generic API — no pyLDAvis.sklearn needed)
# -----------------------
pyLDAvis.enable_notebook()

topic_term_dists = lda.components_ / lda.components_.sum(axis=1)[:, None]
doc_topic_dists = doc_topic
doc_lengths = np.asarray(X.sum(axis=1)).ravel()
term_frequency = np.asarray(X.sum(axis=0)).ravel()

vis = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    sort_topics=True
)

display(vis)

In [ ]:


import os
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────────
n_topics = 12
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# Colors
BG = '#FDF8F3'
TEXT = '#121212'
GRAY = '#666666'
LIGHT = '#999999'
COLORS = ['#D64045','#1A5F7A','#E07A5F','#3D405B','#81B29A','#F4A261',
          '#2A9D8F','#E9C46A','#264653','#A8DADC','#457B9D','#9B5DE5']

# ─────────────────────────────────────────────────────────────────────────────────
# 1. NAME TOPICS WITH OPENAI
# ─────────────────────────────────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)
topic_names = {}

for t in range(n_topics):
    qs = df_docs[df_docs["top_topic"] == t].nlargest(10, "top_topic_conf")["text_raw"].tolist()
    if not qs:
        topic_names[t] = f"Tema {t}"
        continue

    prompt = f"""Estas son preguntas de usuarios. Dame un nombre CORTO (2-4 palabras) como cocina, programación, etc para el tema, en español:
{chr(10).join([f'- {q[:150]}' for q in qs])}

Responde SOLO el nombre del tema."""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30
    )
    topic_names[t] = resp.choices[0].message.content.strip().strip('"')
    print(f"Topic {t}: {topic_names[t]}")

# ─────────────────────────────────────────────────────────────────────────────────
# 2. CREATE VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────────
plt.rcParams['font.family'] = ['Liberation Serif', 'DejaVu Serif', 'serif']

fig = plt.figure(figsize=(22, 28), facecolor=BG)
fig.suptitle('¿Qué Preguntan los Usuarios a la IA?', fontsize=42, fontweight='bold', color=TEXT, y=0.98)
fig.text(0.5, 0.965, f'{len(df_docs):,} preguntas · {n_topics} temas · LDA', ha='center', fontsize=14, color=GRAY, style='italic')

line1 = Line2D([0.05, 0.95], [0.955, 0.955], transform=fig.transFigure, color=TEXT, linewidth=2)
line2 = Line2D([0.05, 0.95], [0.952, 0.952], transform=fig.transFigure, color=TEXT, linewidth=0.5)
fig.add_artist(line1)
fig.add_artist(line2)

gs = fig.add_gridspec(4, 3, left=0.04, right=0.96, top=0.94, bottom=0.03, wspace=0.12, hspace=0.18)

for t in range(n_topics):
    ax = fig.add_subplot(gs[t // 3, t % 3])
    ax.set_facecolor(BG)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

    docs = df_docs[df_docs["top_topic"] == t].nlargest(8, "top_topic_conf")
    color = COLORS[t % len(COLORS)]
    count = len(df_docs[df_docs["top_topic"] == t])

    # Header
    ax.add_patch(FancyBboxPatch((0, 9.0), 0.25, 0.9, boxstyle="round,pad=0.01,rounding_size=0.05", facecolor=color, edgecolor='none'))
    ax.text(0.55, 9.45, f'{t+1:02d}', fontsize=13, fontweight='bold', color=LIGHT, va='center')
    ax.text(1.2, 9.45, topic_names[t], fontsize=16, fontweight='bold', color=TEXT, va='center')
    ax.text(10, 9.45, f'{count} preg.', fontsize=9, color=LIGHT, va='center', ha='right', style='italic')
    ax.plot([0, 10], [8.75, 8.75], color='#E5E5E5', linewidth=1.5)

    # Questions
    y = 8.3
    for _, row in docs.iterrows():
        if y < 0.3: break
        q = str(row["text_raw"]).replace('\n', ' ')[:100]
        wrapped = textwrap.fill(q, width=48).split('\n')[:2]
        ax.plot(0.15, y - 0.15, 'o', markersize=5 + row["top_topic_conf"] * 3, color=color, alpha=0.7)
        for i, line in enumerate(wrapped):
            ax.text(0.5, y - i * 0.4, line + ('...' if i == 1 and len(q) > 80 else ''),
                   fontsize=9.5, color=TEXT if i == 0 else GRAY, va='top')
        y -= 1.0

fig.text(0.5, 0.008, 'Fuente: OASST1 · LDA · OpenAI', ha='center', fontsize=11, color=LIGHT)
fig.savefig('topics_nyt.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Saved: topics_nyt.png")

In [ ]:
df_docs.head()

In [ ]:
df_topics[df_topics["topic_id"] == 0]